# PilotGPT v3: VFR & IFR flight assistant

> **Not for navigation or flight planning.** Educational project. Use official FAA sources,
> current charts and a certificated weather briefing for real flights.

A LangGraph ReAct agent with tools that do the flight math the FAA way and read live weather
from the Aviation Weather Center. The math lives in `src/aviation.py` and is checked against
hand-worked examples in `tests/` (`uv run pytest -q`).

**Setup:** the repo's `.env` sets the LLM endpoint (LM Studio with Qwen works; see `.env.example`).
Live weather needs no key. `SERPAPI_KEY` is optional and only enables a general web-search tool.

In [1]:
# Cell 1 — Setup
import os, re, sys, json
from datetime import date
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_BASE_URL") and os.getenv("OPENAI_API_KEY", "lm-studio") == "lm-studio":
    raise RuntimeError("No LLM endpoint: set OPENAI_BASE_URL (local server) or a real OPENAI_API_KEY in the repo's .env")
os.environ.setdefault("OPENAI_API_KEY", "lm-studio")

PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(PROJECT_DIR / "src"))

import pandas as pd
import folium
import aviation as av
import weather as wx
from airports import AIRPORTS, AIRSPACE

print(f"LLM: {os.getenv('LLM_MODEL', 'gpt-4o-mini')} | web search: {'on' if os.getenv('SERPAPI_KEY') else 'off (no SERPAPI_KEY)'}")

LLM: qwen/qwen3.6-35b-a3b | web search: on


## The flight math, worked examples

In [2]:
# Cell 2 — Wind triangle and magnetic heading: KBOS to KJFK
dep, arr = AIRPORTS["KBOS"], AIRPORTS["KJFK"]
dist = av.haversine_nm(dep["lat"], dep["lon"], arr["lat"], arr["lon"])
tc = av.true_course(dep["lat"], dep["lon"], arr["lat"], arr["lon"])
var = av.magnetic_variation(dep["lat"], dep["lon"])
s = av.wind_triangle(tas_kt=110, true_course_deg=tc, wind_from_deg=250, wind_kt=20, variation_deg=var)

print(f"Distance {dist:.0f} NM, true course {tc:03.0f}°, variation {av.format_variation(var)} (WMM2025)")
print(f"WCA {s.wind_correction_angle:+.1f}° -> true heading {s.true_heading:03.0f}°, magnetic heading {s.magnetic_heading:03.0f}°")
print(f"Headwind {s.headwind:+.0f} kt, crosswind {s.crosswind:+.0f} kt, ground speed {s.ground_speed:.0f} kt, "
      f"time en route {dist / s.ground_speed * 60:.0f} min")

Distance 162 NM, true course 231°, variation 13.9°W (WMM2025)
WCA +3.4° -> true heading 234°, magnetic heading 248°
Headwind +19 kt, crosswind +6 kt, ground speed 91 kt, time en route 107 min


In [3]:
# Cell 3 — Holding entries (AIM 5-3-8) and density altitude
cases = [(360, 360), (360, 100), (360, 200), (90, 200), (180, 45), (180, 300)]
holds = pd.DataFrame([{"inbound_course": c, "heading": h, **av.holding_entry(c, h)} for c, h in cases])
holds.to_csv(OUTPUT_DIR / "holding_entries.csv", index=False)
display(holds[["inbound_course", "heading", "entry", "steps"]])

da = pd.DataFrame([{"airport": a, "oat_c": t, "altimeter_inhg": alt, **av.density_altitude(AIRPORTS[a]["elev_ft"], t, alt)}
                   for a, t, alt in [("KBOS", 28, 29.85), ("KORH", 32, 29.80), ("KATL", 35, 29.92)]])
display(da)

,inbound_course,heading,entry,steps
0,360,360,Direct,Cross the fix and turn right to the outbound h...
1,360,100,Parallel,Cross the fix and turn to the outbound heading...
2,360,200,Teardrop,Cross the fix and fly heading 150° (30° into t...
3,90,200,Parallel,Cross the fix and turn to the outbound heading...
4,180,45,Teardrop,Cross the fix and fly heading 330° (30° into t...
5,180,300,Parallel,Cross the fix and turn to the outbound heading...


,airport,oat_c,altimeter_inhg,pressure_altitude_ft,isa_temp_c,density_altitude_ft
0,KBOS,28,29.85,89,14.8,1654
1,KORH,32,29.80,1129,12.8,3414
2,KATL,35,29.92,1026,13.0,3643


In [4]:
# Cell 4 — Day VFR go/no-go (14 CFR 91.155)
scenarios = [
    ("Good day, Class B", 10, None, 2500, "B"),
    ("4 SM, 2,500 ft ceiling, cruising 3,000 ft, Class C", 4, 2500, 3000, "C"),
    ("Same weather, cruising 2,000 ft", 4, 2500, 2000, "C"),
    ("2 SM in Class D", 2, 3000, 1500, "D"),
    ("1.5 SM, Class G at 1,000 ft AGL", 1.5, None, 1000, "G"),
]
rows = []
for label, vis, ceil, cruise, cls in scenarios:
    r = av.vfr_check(vis, ceil, cruise, cls, surface_area=cls != "G")
    rows.append({"scenario": label, "category": r["flight_category"], "verdict": r["verdict"],
                 "why": " ".join(r["problems"]) or "All minimums met."})
vfr = pd.DataFrame(rows)
vfr.to_csv(OUTPUT_DIR / "vfr_scenarios.csv", index=False)
display(vfr)

,scenario,category,verdict,why
0,"Good day, Class B",VFR,GO: VFR minimums met,All minimums met.
1,"4 SM, 2,500 ft ceiling, cruising 3,000 ft, Cla...",MVFR,NO-GO: below VFR minimums for this plan,"Class C needs 500 ft below clouds: with a 2,50..."
2,"Same weather, cruising 2,000 ft",MVFR,LEGAL but MVFR: minimums met with little margi...,All minimums met.
3,2 SM in Class D,IFR,NO-GO: below VFR minimums for this plan,Visibility 2 SM is below the 3 SM minimum for ...
4,"1.5 SM, Class G at 1,000 ft AGL",IFR,LEGAL but IFR: minimums met with little margin...,All minimums met.


In [5]:
# Cell 5 — Route map (OpenStreetMap base)
m = folium.Map(location=[42.0, -72.0], zoom_start=7, tiles="OpenStreetMap")
colors = {"B": "blue", "C": "purple", "D": "cadetblue", "E": "green", "G": "gray"}
for icao, a in AIRPORTS.items():
    folium.Marker([a["lat"], a["lon"]], tooltip=f"{icao}: {a['name']} (Class {a['class']}, {a['elev_ft']} ft)",
                  icon=folium.Icon(color=colors[a["class"]], icon="plane", prefix="fa")).add_to(m)
folium.PolyLine([[dep["lat"], dep["lon"]], [arr["lat"], arr["lon"]]], color="red", weight=3,
                tooltip=f"KBOS to KJFK: {dist:.0f} NM, TC {tc:03.0f}°").add_to(m)
m.save(str(OUTPUT_DIR / "route_map.html"))
m

In [6]:
# Cell 6 — Live weather from aviationweather.gov
try:
    metar = wx.get_metar("KBOS")
    print(metar.get("raw") or metar)
    if "error" not in metar:
        print(f"Ceiling: {metar['ceiling_ft_agl'] or 'none'} | visibility {metar['visibility_sm']} SM | "
              f"category {av.flight_category(metar['visibility_sm'] or 0, metar['ceiling_ft_agl'])}")
except Exception as e:
    print(f"Live weather unavailable ({e}); the agent will say so instead of guessing.")

METAR KBOS 051954Z 22017G27KT 10SM -RA FEW032 SCT100 OVC140 22/16 A2986 RMK AO2 PK WND 23027/1944 RAB44 SLP112 P0000 TO2220161
Ceiling: 14000 | visibility 10.0 SM | category VFR


## The agent

Every number the agent reports comes from a tool. The system prompt tells it to use tools for
math and weather, never to invent observations, and to end with the not-for-navigation note.

In [7]:
# Cell 7 — Tools and agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
import warnings
warnings.filterwarnings("ignore", message="create_react_agent has been moved")  # still supported in LangGraph 1.x
from langgraph.prebuilt import create_react_agent

def llm_budget(n): return n + int(os.getenv("LLM_REASONING_TOKENS", "0"))
def llm_text(text): return re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()

def _airport(icao):
    a = AIRPORTS.get(icao.strip().upper())
    if not a:
        raise ValueError(f"{icao} is not in the demo airport table: {', '.join(AIRPORTS)}")
    return a

@tool
def airport_info(icao: str) -> str:
    """Airport name, elevation, airspace class with its entry rules, and magnetic variation."""
    a = _airport(icao)
    return json.dumps({"icao": icao.upper(), **a, "airspace_rules": AIRSPACE[a["class"]],
                       "magnetic_variation": av.format_variation(av.magnetic_variation(a["lat"], a["lon"]))})

@tool
def plan_leg(departure: str, destination: str, tas_kt: float, wind_from_deg: float, wind_kt: float) -> str:
    """Distance, true course, wind correction, magnetic heading, ground speed and time en route between two airports."""
    d, a = _airport(departure), _airport(destination)
    dist = av.haversine_nm(d["lat"], d["lon"], a["lat"], a["lon"])
    tc = av.true_course(d["lat"], d["lon"], a["lat"], a["lon"])
    var = av.magnetic_variation(d["lat"], d["lon"])
    s = av.wind_triangle(tas_kt, tc, wind_from_deg, wind_kt, var)
    th, var_whole = round(s.true_heading) % 360, round(var)
    return json.dumps({  # whole degrees, as on a navigation log: MH = TH + West variation (or - East)
        "distance_nm": round(dist, 1), "true_course": round(tc) % 360, "wind_correction_angle": round(s.wind_correction_angle),
        "true_heading": th, "variation": f"{abs(var_whole)}°{'E' if var_whole >= 0 else 'W'}",
        "magnetic_heading": (th - var_whole) % 360, "ground_speed_kt": round(s.ground_speed),
        "headwind_kt": round(s.headwind), "crosswind_kt": round(s.crosswind),
        "time_en_route_min": round(dist / s.ground_speed * 60)})

@tool
def check_vfr(visibility_sm: float, ceiling_ft_agl: float, cruise_ft_agl: float, airspace_class: str,
              in_surface_area: bool = True) -> str:
    """Day VFR go/no-go against 14 CFR 91.155. Use ceiling_ft_agl = 0 when there is no ceiling."""
    return json.dumps(av.vfr_check(visibility_sm, ceiling_ft_agl or None, cruise_ft_agl, airspace_class,
                                   surface_area=in_surface_area))

@tool
def holding_entry(inbound_course: float, heading: float, right_turns: bool = True) -> str:
    """Recommended holding entry (direct, teardrop or parallel) from the heading at the fix."""
    return json.dumps(av.holding_entry(inbound_course, heading, right_turns))

@tool
def density_altitude(icao: str, oat_c: float, altimeter_inhg: float = 29.92) -> str:
    """Pressure and density altitude at an airport from temperature and altimeter setting."""
    a = _airport(icao)
    return json.dumps({"icao": icao.upper(), "airport": a["name"], "field_elevation_ft": a["elev_ft"],
                       **av.density_altitude(a["elev_ft"], oat_c, altimeter_inhg)})

@tool
def live_metar(icao: str) -> str:
    """Latest decoded METAR from aviationweather.gov (official). Includes ceiling and flight category."""
    try:
        m = wx.get_metar(icao)
        if "error" not in m and m["visibility_sm"] is not None:
            m["flight_category"] = av.flight_category(m["visibility_sm"], m["ceiling_ft_agl"])
        return json.dumps(m)
    except Exception as e:
        return json.dumps({"error": f"Live weather unavailable: {e}"})

@tool
def live_taf(icao: str) -> str:
    """Latest TAF (forecast) from aviationweather.gov, as raw text."""
    try:
        return json.dumps(wx.get_taf(icao))
    except Exception as e:
        return json.dumps({"error": f"TAF unavailable: {e}"})

tools = [airport_info, plan_leg, check_vfr, holding_entry, density_altitude, live_metar, live_taf]

if os.getenv("SERPAPI_KEY"):
    @tool
    def web_search(query: str) -> str:
        """General web search for background only (news, airport facilities). Never use for weather, NOTAMs or TFRs."""
        try:
            return json.dumps(wx.web_search(query))
        except Exception as e:
            return json.dumps({"error": str(e)})
    tools.append(web_search)

SYSTEM = (
    "You are PilotGPT, a flight-planning study assistant for student and private pilots. "
    "Use the tools for every calculation, weather observation and airspace fact; never estimate numbers yourself "
    "or invent weather. Show the key numbers and say which rule they come from. If a tool returns an error, say so. "
    "Use names, sides, directions and options exactly as the tools give them. Don't add rules, sectors, alternatives "
    "or facts the tools didn't return; if a tool lists options, offer only those. Weather and NOTAMs/TFRs must come from official sources. End every answer with: "
    "'Not for navigation: verify with official FAA sources and a weather briefing.'"
)
llm = ChatOpenAI(model=os.getenv("LLM_MODEL", "gpt-4o-mini"), temperature=0, max_tokens=llm_budget(1200))
pilotgpt = create_react_agent(llm, tools=tools, prompt=SYSTEM)
print(f"PilotGPT ready: {len(tools)} tools -> {[t.name for t in tools]}")

PilotGPT ready: 8 tools -> ['airport_info', 'plan_leg', 'check_vfr', 'holding_entry', 'density_altitude', 'live_metar', 'live_taf', 'web_search']


In [8]:
# Cell 8 — Ask PilotGPT
queries = [
    "I'm planning a VFR flight into KBDL. What airspace is it and what do I need before entering?",
    "Visibility 4 SM, ceiling 2,500 ft AGL, and I'd cruise at 3,000 ft AGL near KBDL. Can I go VFR?",
    "Plan KBOS to KJFK: TAS 110 kt, winds from 250 at 20 kt. Give me magnetic heading, ground speed and time en route.",
    "What's the density altitude at KORH with OAT 32°C and altimeter 29.80?",
    "Holding: inbound course 090, standard right turns, I'm heading 200 at the fix. Which entry and how do I fly it?",
    "Is Boston Logan VFR right now? Use the live METAR.",
]
runs = []
for q in queries:
    result = pilotgpt.invoke({"messages": [{"role": "user", "content": q}]})
    answer = llm_text(result["messages"][-1].content)
    calls = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
    runs.append({"question": q, "tools_called": calls, "answer": answer})
    print(f"\nPilot: {q}\nTools: {calls}\nPilotGPT: {answer}\n" + "-" * 60)


Pilot: I'm planning a VFR flight into KBDL. What airspace is it and what do I need before entering?
Tools: ['airport_info']
PilotGPT: Based on the official airport data for KBDL (Hartford Bradley):

**Airspace Class:** Class C

**Requirements to enter:**
* **Radio:** Two-way radio contact established before entry
* **Equipment:** Transponder with Mode C / ADS-B Out
* **Altitude/Geography:** Surface to about 4,000 ft AGL (5 NM core, 10 NM shelf)

Not for navigation: verify with official FAA sources and a weather briefing.
------------------------------------------------------------

Pilot: Visibility 4 SM, ceiling 2,500 ft AGL, and I'd cruise at 3,000 ft AGL near KBDL. Can I go VFR?
Tools: ['airport_info', 'check_vfr']
PilotGPT: Based on the VFR minimums check for Class C airspace near KBDL:

**Verdict:** NO-GO: below VFR minimums for this plan.

**Key Numbers & Rules:**
* **Cloud Clearance:** Class C airspace requires remaining 500 feet below the clouds. With a 2,500 ft ceiling, you c

In [9]:
# Cell 9 — Save the run, labelled with the model that produced it
json.dump({"model": os.getenv("LLM_MODEL", "qwen/qwen3.6-35b-a3b"), "date": date.today().isoformat(), "runs": runs},
          open(OUTPUT_DIR / "agent_run.json", "w"), indent=2, ensure_ascii=False)
pd.DataFrame(AIRPORTS).T.to_csv(OUTPUT_DIR / "airport_database.csv")
print("Saved outputs/agent_run.json, holding_entries.csv, vfr_scenarios.csv, route_map.html, airport_database.csv")

Saved outputs/agent_run.json, holding_entries.csv, vfr_scenarios.csv, route_map.html, airport_database.csv
